<a href="https://colab.research.google.com/github/Seif-Abouelkhair/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-Abouelkhair/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

For this lane, the final modeling unit is one content item for one pseudonymized client (client_hash_id × content_hash_id).

The source table, fact_content_daily_performance, is at a daily grain: one row represents one client × content item × report date. We aggregate those daily observations into a content-level feature frame.

I will use two non-overlapping monthly windows:

February 2026: feature window. These are the signals that would be available at the decision point on February 28.
March 2026: label window. This contains the subsequent observed outcome that the ranking is intended to prioritize.

The two windows are deliberately separated so that information from the outcome period does not enter the features.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np
def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Enter your Hugging Face READ token: ")
HF_TOKEN = get_hf_token()
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Label window: March 2026")

Enter your Hugging Face READ token: ··········
Connected to FlyRank warehouse.
Feature window: February 2026
Label window: March 2026


## 2. Fields: feature / label / context / excluded

Features

I will build a small feature frame from the February 2026 window using five features:

gsc_impressions_feb — total measured GSC impressions during February. Available when the decision is made because it summarizes the completed feature window.
gsc_clicks_feb — total measured GSC clicks during February. Available when the decision is made because it uses only completed February observations.
avg_position_feb — impression-weighted average GSC position during February. Available when the decision is made because it is calculated only from the feature window.
content_age_days — content age at the feature snapshot. Available when the decision is made because it describes the content before the March outcome window.
days_since_last_update — days since the most recent content update at the feature snapshot. Available when the decision is made and represents content freshness before the outcome period.
Label

The future label is went_dark: a content item with measured March GSC data and zero March clicks.

The label is calculated from the March outcome window and will never be used as a feature.

Context

client_hash_id, content_hash_id, and report_date are context fields. They are used for grouping, joining, filtering, and time-aware splitting, but not as predictive features.

Excluded

March performance fields are excluded from the feature set because they occur after the February decision point. Label-derived fields such as went_dark are also excluded because using them as features would leak the outcome into the model.

I will also exclude rows without measured GSC data from the March outcome calculation rather than treating missing data as zero clicks.

In [2]:
schema = con.sql(
    f"DESCRIBE SELECT * FROM {FEB} LIMIT 1"
).df()
print("February table columns:")
display(schema[["column_name", "column_type"]])

February table columns:


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# QUERY 1 — Grain verification
grain_check = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {FEB}
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    LIMIT 5
    """
).df()

print("Duplicate client × content × day groups:", len(grain_check))
display(grain_check)

Duplicate client × content × day groups: 0


,client_hash_id,content_hash_id,report_date,row_count


In [5]:
# QUERY 2 — Count and date window verification
window_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS contents
    FROM {FEB}
    """
).df()

display(window_check)

,row_count,min_date,max_date,clients,contents
0,7355108,2026-02-01,2026-02-28,54,321546


In [6]:
# QUERY 3 — GSC availability verification
availability_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS gsc_not_available_or_unknown
    FROM {FEB}
    """
).df()

display(availability_check)

,total_rows,gsc_available_rows,gsc_not_available_or_unknown
0,7355108,2621783,4733325


In [8]:
feature_notes = pd.DataFrame({
    "feature": [
        "gsc_impressions_feb",
        "gsc_clicks_feb",
        "avg_position_feb",
        "content_age_days",
        "days_since_last_update",
    ],
    "available_when": [
        "At the February 28 decision point",
        "At the February 28 decision point",
        "At the February 28 decision point",
        "At the February 28 decision point",
        "At the February 28 decision point",
    ]
})

display(feature_notes)

,feature,available_when
0,gsc_impressions_feb,At the February 28 decision point
1,gsc_clicks_feb,At the February 28 decision point
2,avg_position_feb,At the February 28 decision point
3,content_age_days,At the February 28 decision point
4,days_since_last_update,At the February 28 decision point


In [9]:
march_label = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS gsc_clicks_mar,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS measured_days_mar
    FROM {MAR}
    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

# Keep only content items with at least one measured March day.
march_label = march_label[march_label["measured_days_mar"] > 0].copy()

march_label["went_dark"] = (
    march_label["gsc_clicks_mar"] == 0
).astype(int)

display(march_label.head())
print("March label rate:", round(march_label["went_dark"].mean(), 3))

,client_hash_id,content_hash_id,gsc_clicks_mar,measured_days_mar,went_dark
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,0.0,29,1
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,1.0,16,0
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,1.0,31,0
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,0.0,17,1
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,0.0,30,1


March label rate: 0.611


In [11]:
march_label = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS gsc_clicks_mar,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS measured_days_mar
    FROM {MAR}
    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

# Keep only content items with at least one measured March day.
march_label = march_label[march_label["measured_days_mar"] > 0].copy()

march_label["went_dark"] = (
    march_label["gsc_clicks_mar"] == 0
).astype(int)

display(march_label.head())
print("March label rate:", round(march_label["went_dark"].mean(), 3))

,client_hash_id,content_hash_id,gsc_clicks_mar,measured_days_mar,went_dark
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2.0,31,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,0.0,26,1
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,0.0,30,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,6.0,31,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,16.0,31,0


March label rate: 0.611


## 4. Data limits

This contract supports a directional, decision-support analysis, but it cannot establish causality.

First, the warehouse is an unbalanced panel: different clients have different amounts of historical search data. A page with limited history should not automatically be treated as equivalent to a page with a long observed history.

Second, missing GSC availability must not be interpreted as zero search performance. I therefore use the availability flag and require measured data for the March outcome rather than converting unavailable observations into zero clicks.

Third, the March label represents one defined future observation window. It can be used to evaluate whether a page met the chosen outcome during that window, but it cannot prove that the page will decline indefinitely.

Finally, this analysis does not prove that a search-engine algorithm caused any observed movement. The eventual ranking is intended to support content-review prioritization, not to establish causal effects.

In [15]:
print("Contract limitation check:")
print("- Feature window: February 2026")
print("- Outcome window: March 2026")
print("- Final June 2026 month remains reserved for later testing")
print("- Missing GSC observations are not treated as zero performance")
print("- Results will be interpreted as decision-support, not causal proof")

Contract limitation check:
- Feature window: February 2026
- Outcome window: March 2026
- Final June 2026 month remains reserved for later testing
- Missing GSC observations are not treated as zero performance
- Results will be interpreted as decision-support, not causal proof


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.